# Week 2 Day 1 Lab - Evals and Observability DeepEval

## Measuring an agent on UnderCover's harder claims

A new batch of 25 claims has arrived at UnderCover, and they're harder than the queue from Week 1. Many have long lists of items, with payments from airlines, hotels and other insurers to take off, and deductibles, limits and caps to apply.

We trace the agent from Week 1 with DeepEval and score its decisions against the handlers'. Then we run two experiments: the agent working out each payout itself, and the agent with a payout tool.

## 4 parts:

PART 1: The harder claims

PART 2: Tracing the agent with DeepEval

PART 3: Scoring an agent

PART 4: Experiments in DeepEval

In [8]:
import json
import os
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.evaluate.configs import AsyncConfig
from deepeval.integrations.langchain import CallbackHandler
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase
from deepeval.tracing import observe, update_current_trace
from deepeval.tracing.context import current_span_context, current_trace_context
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made in Week 1 in the `undercover` folder.

In [9]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

OpenRouter API key found


## Part 1: The harder claims

The harder claims are in `data/evals/harder`, with their own policy records and the handlers' decisions. The policy wording is the same as before.

In [10]:
HARDER = Path("../data/evals/harder")


def load(name):
    """Read one of the JSON files of the harder claims."""
    return json.loads((HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

25 claims, 25 labels, 25 policies


In [11]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "items": len(claim["items"]),
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,items,amount_claimed,decision,payout
0,CLM-1201,Suitcase lost on the way to Lisbon,16,1725.77,approve,553.28
1,CLM-1202,Equipment case crushed in the hold on the way ...,9,7357.37,approve,2048.44
2,CLM-1203,Rented apartment in Rome broken into,14,6207.43,approve,2910.44
3,CLM-1204,Safari cut short when my sister was badly hurt...,11,5785.45,approve,4787.25
4,CLM-1205,Family of four held overnight at Newark by an ...,14,1681.14,approve,1021.14
5,CLM-1206,Golf clubs stolen from the trunk of a rental car,13,2587.96,approve,1450.01
6,CLM-1207,Backpack pulled off my chair at a cafe in Barc...,14,3581.17,approve,1306.72
7,CLM-1208,Torn knee ligament on the Routeburn Track,16,4518.80,approve,3668.80
8,CLM-1209,Broken wrist and a head wound when I fell out ...,16,5100.15,approve,4180.15
9,CLM-1210,Amalfi trip canceled after my cousin's heart a...,4,4796.70,decline,0.00


Similar mix of decisions

In [5]:
data["decision"].value_counts()

decision
approve    14
refer       6
decline     5
Name: count, dtype: int64

### A burglary in Rome

Here's one of the longer claims.

In [6]:
claims

[{'claim_id': 'CLM-1201',
  'policy_id': 'POL-2201',
  'customer_id': 'CUS-3201',
  'claim_type': 'baggage',
  'submitted_on': '2026-08-31',
  'incident_date': '2026-08-04',
  'incident_location': 'Lisbon, Portugal',
  'summary': 'Suitcase lost on the way to Lisbon',
  'story': 'My suitcase did not arrive in Lisbon on August 4. Skylark searched for three weeks, declared it lost and paid me $800. I have listed everything in it at what I paid, with receipts for everything over $100.',
  'items': [{'description': 'Hard-shell suitcase',
    'category': 'luggage',
    'amount': 239.99},
   {'description': 'Linen dress', 'category': 'clothing', 'amount': 184.5},
   {'description': 'Two linen shirts',
    'category': 'clothing',
    'amount': 139.98},
   {'description': 'Sandals, one pair',
    'category': 'clothing',
    'amount': 94.95},
   {'description': 'Walking shoes, one pair',
    'category': 'clothing',
    'amount': 159.0},
   {'description': 'Straw hat', 'category': 'accessories', 

In [17]:
rome = [claim for claim in claims if claim["claim_id"] == "CLM-1203"][0]

print(rome["summary"])
print(rome["story"])
pd.DataFrame(rome["items"])

Rented apartment in Rome broken into
On July 22 Erik and I came back to our rented apartment in Trastevere after dinner to find the front door forced and our things gone. We called the police at once. The apartment owner's insurer has paid us $2,250. Receipts for everything over $100 are attached.


,description,category,amount
0,Laptop,electronics,1899.00
1,Tablet,electronics,649.00
2,Smartwatch,electronics,399.00
3,E-reader,electronics,159.99
4,Noise-canceling headphones,electronics,549.00
5,Handheld game console,electronics,599.99
6,VR headset,electronics,649.00
7,Portable speaker,electronics,189.95
8,Leather jacket,clothing,389.00
9,Two pairs of sneakers,clothing,258.00


This is how the handler worked out the payout.

In [12]:
decision = labels["CLM-1203"]

print(decision["reason"])
pd.DataFrame(decision["calculation"])

A locked apartment is not a place the public can get to, so the belongings were not unattended, and the police were called that night. Electronics Plus raises the limit to $1,200 for the laptop, and the tablet, the smartwatch and the e-reader are paid as claimed. Headphones, a game console and a VR headset are not on the Electronics Plus list and count at the $500 per-item limit on First. Cash is covered up to $100 in all. The items come to $5,160.44. The owner's insurer's $2,250 comes off, with no deductible on First. The payout of $2,910.44 is inside the $3,000 maximum and needs sign-off.


,step,amount
0,"Laptop, limited to $1,200.00",1200.00
1,Tablet,649.00
2,Smartwatch,399.00
3,E-reader,159.99
4,"Noise-canceling headphones, limited to $500.00",500.00
5,"Handheld game console, limited to $500.00",500.00
6,"VR headset, limited to $500.00",500.00
7,Portable speaker,189.95
8,Leather jacket,389.00
9,Two pairs of sneakers,258.00


## Part 2: Tracing the agent with DeepEval

This is the agent from Week 1 Day 2 with two changes. The customer's policy record goes in the prompt with the claim, so its only tool is `record_decision`. We also turn Luna's reasoning off with `reasoning_effort="none"`, which makes each call faster and cheaper.

In [13]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

The tool's docstring is in Google style. With `parse_docstring=True`, LangChain turns each line under `Args` into the description of that argument.

The rationale comes before the decision, so the model writes its reasons before it decides. As in UnderCover's claims system, a decision can't be changed once it's recorded, and an approval needs a payout above zero.

In [14]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [15]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="none", timeout=120, max_retries=5)


baseline_agent = create_agent(model=luna, tools=[record_decision], system_prompt=f"{INSTRUCTIONS}\n{policy_wording}\n")

DeepEval's `CallbackHandler` records a trace of everything LangChain does. We pass it to the agent in the `config`.

The trace goes to Confident AI, DeepEval's platform, when `CONFIDENT_API_KEY` is in your `.env` file. Without a key, DeepEval prints a message and skips the trace. Everything else in this lab still works.

In [18]:
recorded.clear()
result = baseline_agent.invoke(
    {"messages": [{"role": "user", "content": make_user_prompt(rome)}]},
    config={"callbacks": [CallbackHandler()]},
)

for message in result["messages"][1:]:
    message.pretty_print()

[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

================================== Ai Message ==================================
Tool Calls:
  record_decision (call_ZL8QpPiCpS1L6v1OWDse46oF)
 Call ID: call_ZL8QpPiCpS1L6v1OWDse46oF
  Args:
    claim_id: CLM-1203
    rationale: Approve the documented theft: it occurred during the covered trip, was reported to local police, and the police report and purchase receipts are attached. Under sections 5 and 8, Electronics Plus raises the per-item limit for covered electronics to $1,200; the remaining items use the $500 First-plan per-item limit, pairs of sneakers count as one item, and cash is subject to the $100 cap. Applying section 10, eligible items total $5,311.43, less the owner's insurer payment of $2,250 and the $0 First-plan deductible, yielding $3,061.43, within the $3,000 baggage benefit maximum; payout is capped at $3,000.
    decision: approve
    payout: 3000
================================= Tool Message =================================
Name: record_decision

Recorded approve

In [27]:
result['messages'][-1]

AIMessage(content='**Approved — $3,000.** The theft is supported by the police report and receipts and occurred during the covered trip. After applying the per-item limits, the $2,250 payment from the apartment owner’s insurer and the First plan’s $3,000 baggage maximum, the payable amount is capped at **$3,000**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 3368, 'total_tokens': 3442, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 200, 'cached_tokens': 3165, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 9.395e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 9.395e-05, 'upstream_inference_prompt_cost': 5.695e-05, 'upstream_inference_completions_cost': 3.7e-05}}, 'model_pro

If you use Confident AI, open **Observability** and then **Traces**. The trace shows each call to the model with its prompt, its reply and the time it took, and the tool call in between.

## Part 3: Scoring an agent

In Week 1 a claim was right when the decision matched the handler's and the payout was within a dollar. We still count that, but not every mistake costs UnderCover the same.

### The confusion matrix

A confusion matrix compares an agent's decisions with the right answers. Each row is the handler's decision and each column is the agent's. The claims on the diagonal are the ones where they agree, and each cell off the diagonal is a kind of mistake:

- **A wrongful decline:** the agent declines a claim the handler approved or referred. The customer is let down and may complain.
- **A wrongful payment:** the agent approves a claim the handler declined or referred. UnderCover pays money it doesn't owe.
- **A needless referral:** the agent refers a claim the handler decided. A handler has to spend time on it.

The confusion matrix compares decisions only, so an approval with the wrong payout still sits on the diagonal. The cost of mistakes below counts the wrong amounts.

### Precision and recall

Precision and recall come from the same matrix, with an approval as the positive:

- **Precision:** of the claims the agent approved, the share the handler approved.
- **Recall:** of the claims the handler approved, the share the agent approved.

### The cost of mistakes

A cost matrix gives each kind of mistake a cost in dollars. A wrongful payment costs the amount paid, and a wrong amount on a correct approval costs the difference.

In [19]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">The UnderCover business has more tolerance for false positives (we cover someone when we shouldn't) than false negatives (we decline someone when we should have covered them), because false negatives damage our reputation and customer loyalty. Our evals take this into account.<br/><br/>
            Agree the figures in the cost matrix with the business before you run your evals.
            </span>
        </td>
    </tr>
</table>

### Kappa

Accuracy can flatter an agent. 14 of the 25 claims are approvals, so an agent that approves everything gets 56% of the decisions right. Cohen's kappa corrects for this. It measures how much more often the agent agrees with the handlers than it would by chance, based on how often each of them picks each decision. A kappa of 1 is perfect agreement and 0 is no better than chance.

In [ ]:
def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


approve_everything = pd.DataFrame({"handler": data["decision"], "agent": "approve"})

print(f"Approve everything gets a kappa of {kappa(approve_everything):.2f}")

Approve everything gets a kappa of 0.00


Here's the confusion matrix for approving everything. Every claim lands in the approve column.

In [ ]:
def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()


show_confusion_matrix(approve_everything, "Approve everything")

### The straight-through rate

The straight-through rate is the share of claims the agent decides without referring them to a handler. It includes approvals over $1,000, which still need a handler's sign-off. An agent that refers every claim never pays or declines a claim wrongly. But its straight-through rate is 0%, and each of its 19 needless referrals costs $50. The handlers refer 6 of the 25 claims, so their rate is 76%.

## Part 4: Experiments in DeepEval

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Goldens, test cases and metrics</h2>
            <span style="color:#118AB2;">A DeepEval <b>golden</b> is a test input with its expected output, and a dataset is a list of goldens. To score an agent, you run it on each golden and put its answer in an <b>LLMTestCase</b>. A <b>metric</b> scores a test case, and <code>evaluate</code> runs the metrics on all the test cases and reports the results.<br/><br/>
            DeepEval has ready-made metrics, many of them judged by an LLM. Ours are exact comparisons with the handlers' decisions, so we write a custom metric by subclassing <code>BaseMetric</code>. The scores for a whole run, such as kappa, we work out ourselves in pandas.
            </span>
        </td>
    </tr>
</table>

### The dataset

We make the harder claims into a dataset of goldens, with the handler's decision as the expected output. Each golden is named for its claim id.

In [ ]:
goldens = []

for claim in claims:
    label = labels[claim["claim_id"]]
    goldens.append(Golden(
        name=claim["claim_id"],
        input=json.dumps(claim),
        expected_output=json.dumps({"decision": label["decision"], "payout": label["payout"], "reason": label["reason"]}),
        additional_metadata={"claim_id": claim["claim_id"]},
    ))

len(goldens)

25

In [11]:
TINY = True
N_TIMES = 1

if TINY:
    goldens = goldens[:5]

In [12]:
dataset = EvaluationDataset(goldens=goldens)
len(dataset.goldens)

5

By default we run all 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

`N_TIMES` is the number of times each experiment runs. The agent's answers vary from run to run, and more runs give steadier scores. With more than one run, each confusion matrix adds up the claims from every run, and the comparison at the end averages the scores.

### The task and the metric

The task runs an agent on one claim. If the agent finishes without recording a decision, the claim would go to a handler, so we count it as a referral.

In [38]:
def decide(agent, claim):
    """Run an agent on one claim and return its decision, with the cost of its model calls."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [CallbackHandler()]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "cost": cost}

`RightMetric` is a DeepEval metric. It scores one test case as 1 when the agent got the claim right, and 0 when it didn't. `score_table` puts each claim's answer beside the handler's, and `run_scores` works out the scores for a whole run.

In the run's scores, `decision_accuracy` is the share of claims on the diagonal of the confusion matrix. `accuracy` also needs the payout within a dollar.

In [39]:
class RightMetric(BaseMetric):
    """Test case metric: 1 when the agent's decision matches the handler's and the payout is within a dollar."""

    def __init__(self):
        self.threshold = 1
        self.async_mode = False

    def measure(self, test_case, *args, **kwargs):
        answer = json.loads(test_case.actual_output)
        expected = json.loads(test_case.expected_output)
        self.score = int(is_right(answer, expected))
        self.success = self.score >= self.threshold
        self.reason = f"Agent: {answer['decision']} ${answer['payout']:,.2f}. Handler: {expected['decision']} ${expected['payout']:,.2f}. Cost of mistake: ${mistake_cost(answer, expected):,.2f}"
        return self.score

    async def a_measure(self, test_case, *args, **kwargs):
        return self.measure(test_case)

    def is_successful(self):
        return self.success

    @property
    def __name__(self):
        return "Right"


def score_table(test_cases):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for test_case in test_cases:
        answer = json.loads(test_case.actual_output)
        expected = json.loads(test_case.expected_output)
        rows.append({
            "claim_id": test_case.name,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "llm_cost": answer["cost"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "llm_cost": table["llm_cost"].sum(),
    }

`run_experiment` runs an agent on every golden in the dataset, as many times as `N_TIMES`. It loops with the dataset's `evals_iterator`, which scores each trace with `RightMetric`. The `@observe` decorator on `run_claim` makes each claim one trace. The `CallbackHandler` inside it adds the model calls and the tool call to that trace. `update_current_trace` gives the trace the claim's test case, so the metric has something to score.

DeepEval saves the traces of the latest run. To read them, run `uv run deepeval inspect` in the terminal, in the folder you ran the notebook from (`labs`). Each run replaces the one before, so inspect after each experiment. The claims run one after another, so each run takes about 2 minutes.

In [45]:
json.loads(test_case.expected_output)


{'decision': 'approve',
 'payout': 553.28,
 'reason': "The carrier has declared the bag lost. The suitcase, the dress, the walking shoes and the sunglasses count at the $150 per-item limit on Economy, and the other lines are paid as claimed, so the items come to $1,453.28. The carrier's $800 comes off, then the $100 deductible, leaving $553.28, inside the $750 maximum. The carrier paid more than the maximum, but section 10 takes its payment off before the maximum."}

In [44]:
json.loads(test_case.actual_output)


{'decision': 'refer',
 'payout': 0.0,
 'rationale': "Refer under section 2: a parent's sickness or injury can qualify only when confirmed in writing by a physician. The surgeon's letter has not been provided, so a required document is missing; the attached cancellation confirmations document the nonrefundable trip costs but do not establish the medical reason.",
 'cost': 0.000484805}

In [29]:
@observe(type="agent")
def run_claim(agent, golden):
    """Run an agent on one golden. The trace of this call is scored with the golden's expected output."""
    answer = decide(agent, json.loads(golden.input))
    test_case = LLMTestCase(
        name=golden.name,
        input=golden.input,
        actual_output=json.dumps(answer),
        expected_output=golden.expected_output,
    )
    update_current_trace(name=golden.name, test_case=test_case)
    return test_case


def run_experiment(name, agent):
    """Run an agent on the dataset N_TIMES, score each run with DeepEval, and return the claims of every run in one table."""
    tables = []
    for run in range(1, N_TIMES + 1):
        recorded.clear()
        # A trace left open by an earlier cell (the Part 2 demo, or a run that was stopped) would break this run
        current_span_context.set(None)
        current_trace_context.set(None)
        test_cases = []
        for golden in dataset.evals_iterator(
            metrics=[RightMetric()],
            identifier=f"{name}, run {run}",
            async_config=AsyncConfig(run_async=False),
        ):
            test_cases.append(run_claim(agent, golden))
        tables.append(score_table(test_cases).assign(run=run))
    return pd.concat(tables, ignore_index=True)

### Experiment 1: the baseline

The agent works out each payout itself, following section 10 of the policy wording.

In [31]:
#baseline_table = run_experiment("Baseline", baseline_agent)

In [ ]:
show_confusion_matrix(baseline_table, "Baseline")

Here are the claims it got wrong, with the agent's rationale.

In [ ]:
baseline_table[~baseline_table["right"]]

,claim_id,handler,agent,handler_payout,agent_payout,right,mistake_cost,llm_cost,rationale,run
0,CLM-1201,approve,approve,553.28,750.00,False,196.72,0.000558,The carrier report confirms the suitcase was l...,1
1,CLM-1202,approve,approve,2048.44,3000.00,False,951.56,0.000324,Approve the covered physical damage reported b...,1
2,CLM-1203,approve,approve,2910.44,3000.00,False,89.56,0.000333,Approve the documented theft under section 5: ...,1
3,CLM-1204,approve,refer,4787.25,0.00,False,50.00,0.000318,Refer for review. Section 2 covers interruptio...,1
4,CLM-1205,approve,approve,1021.14,961.14,False,60.00,0.000333,Approve eligible reasonable delay expenses: th...,1


### Experiment 2: a payout tool

The payout rules in section 10 are fixed business logic, and business logic belongs in code. `week2day1_payout.py` has UnderCover's payout rules as a Python function, `work_out_payout`, and we give it to the agent as a tool. The agent still judges whether the claim is covered, which lines count and what counts as one item.

In [32]:
@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

A tool is still a Python function, so we can try it ourselves. Here's a suitcase and three T-shirts on an Economy policy, where the airline has paid $100.

In [33]:
calculate_payout.invoke({
    "policy_id": "POL-2201",
    "benefit": "baggage",
    "lines": [
        {"description": "Hard-shell suitcase", "amount": 239.99, "kind": "item"},
        {"description": "Three T-shirts", "amount": 74.97, "quantity": 3, "kind": "item"},
    ],
    "paid_by_others": 100,
})

{'calculation': [{'step': 'Hard-shell suitcase, limited to $150.00',
   'amount': 150.0},
  {'step': 'Three T-shirts', 'amount': 74.97},
  {'step': 'Less what others have paid or refunded', 'amount': -100.0},
  {'step': 'Less the deductible', 'amount': -100.0}],
 'payout': 24.97,
 'needs_sign_off': False}

The agent gets the tool and one more line in its instructions.

In [34]:
PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")

In [19]:
payout_tool_table = run_experiment("Payout tool", payout_agent)

Output()

CLM-1201: approve $553.28

CLM-1202: approve $2,048.44

CLM-1203: approve $2,910.44

CLM-1204: approve $4,787.25

CLM-1205: approve $1,021.14

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1201 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1202 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1203 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1204 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ CLM-1205 (Passed 1 metrics)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric        ┃ Average Score             ┃ Pass Rate                                            ┃ Total       │
│ ━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━ │
│  Right         │ 1.00                      │ 100.00% | passed=5 | failed=0                        │ 5           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=6945165;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 41.94s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 100.0% | Passed: 5 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
show_confusion_matrix(payout_tool_table, "Payout tool")

In [ ]:
payout_tool_table[~payout_tool_table["right"]]

,claim_id,handler,agent,handler_payout,agent_payout,right,mistake_cost,llm_cost,rationale,run


In [ ]:
print("The baseline got either decision or amount wrong for:", len(baseline_table[~baseline_table["right"]]) )
print("The payout tool version got either decision or amount wrong for:", len(payout_tool_table[~payout_tool_table["right"]]) )


The baseline got either decision or amount wrong for: 5
The payout tool version got either decision or amount wrong for: 0


If you use Confident AI, open the traces for these claims to see the lines the agent passed to the tool and the calculation that came back.

### Comparing the experiments

In [ ]:
def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()


pd.DataFrame([average_scores(baseline_table), average_scores(payout_tool_table)], index=["Baseline", "Payout tool"]).round(3)

/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_48321/1996822021.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return (observed - expected) / (1 - expected)


,decision_accuracy,accuracy,precision,recall,kappa,cost_of_mistakes,straight_through,llm_cost
Baseline,0.8,0.0,1.0,0.8,0.0,1347.84,0.8,0.002
Payout tool,1.0,1.0,1.0,1.0,NaN,0.00,1.0,0.003


DeepEval also prints a table for each `evaluate` call, and keeps the results of the latest run in a `.deepeval` folder. On Confident AI, the test runs are under **Evals**, where you can compare them side by side, claim by claim.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: on Week 1 Day 4 we wrote a concise version of the policy wording, in <code>data/policy_wording_short.md</code>. Run a third experiment with the concise wording and the payout tool, and compare it with the full wording. Does it keep the accuracy? Is it cheaper or faster?<br/><br/>
            Then change <code>WRONGFUL_DECLINE</code> and score the runs again. Does it change which wording you would choose?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

In [24]:
label = labels[claims[0]["claim_id"]]
golden = Golden(
        name=claims[0]["claim_id"],
        input=json.dumps(claim),
        expected_output=json.dumps({"decision": label["decision"], "payout": label["payout"], "reason": label["reason"]}),
        additional_metadata={"claim_id": claims[0]["claim_id"]},
    )

In [40]:
recorded.clear()
# A trace left open by an earlier cell (the Part 2 demo, or a run that was stopped) would break this run
current_span_context.set(None)
current_trace_context.set(None)
test_case = run_claim(payout_agent, golden)



CLM-1225: refer $0.00


[Confident AI Trace Log]  No Confident AI API key found. Skipping trace posting.

In [46]:
json.loads(test_case.expected_output)

{'decision': 'approve',
 'payout': 553.28,
 'reason': "The carrier has declared the bag lost. The suitcase, the dress, the walking shoes and the sunglasses count at the $150 per-item limit on Economy, and the other lines are paid as claimed, so the items come to $1,453.28. The carrier's $800 comes off, then the $100 deductible, leaving $553.28, inside the $750 maximum. The carrier paid more than the maximum, but section 10 takes its payment off before the maximum."}

In [47]:
json.loads(test_case.actual_output)

{'decision': 'refer',
 'payout': 0.0,
 'rationale': "Refer under section 2: a parent's sickness or injury can qualify only when confirmed in writing by a physician. The surgeon's letter has not been provided, so a required document is missing; the attached cancellation confirmations document the nonrefundable trip costs but do not establish the medical reason.",
 'cost': 0.000484805}

In [53]:
from deepeval import evaluate
from deepeval import assert_test
assert_test(test_case, [RightMetric()], run_async=True)

Output()

AssertionError: Metrics: Right (score: 0.0, threshold: 1.0, strict: False, error: None, reason: Agent: refer $0.00. Handler: approve $553.28. Cost of mistake: $50.00) failed.